# BODAQS Data Explorer - Study Set Scope

This notebook is a read-only Study Set consumer. It starts from a saved Study Set created by the web app or another Library API client, then builds the standard data explorer widgets from that persisted scope.

This notebook deliberately does not include an in-notebook session selector. To change the scope, edit `TARGET_STUDY_SET_ID` and rerun the setup cells.

Current pilot limitation: the adapter bridge used by notebook widgets supports one library at a time. Use a Study Set whose sessions all belong to `LIBRARY_ID`. Study Set groupings are included as grouped scope entities.


## 1. Configure Library And Study Set

Set `LIBRARIES_ROOT`, `LIBRARY_ID`, and `TARGET_STUDY_SET_ID` before running the rest of the notebook. `TARGET_STUDY_SET_ID` is the Study Set ID saved by the web application.


In [1]:
from pathlib import Path
import sys

from IPython.display import display
import pandas as pd


def find_analysis_dir(start: Path | None = None) -> Path:
    """Find the analysis package root whether Jupyter starts in repo root or analysis/."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "bodaqs_analysis").is_dir():
            return candidate
        if (candidate / "analysis" / "bodaqs_analysis").is_dir():
            return candidate / "analysis"
    raise RuntimeError("Could not find the BODAQS analysis package root from the current working directory.")


ANALYSIS_DIR = find_analysis_dir()
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

LIBRARIES_ROOT = Path(r"C:\Users\benco\OneDrive\BODAQS-data")
LIBRARY_ID = "archie"
TARGET_STUDY_SET_ID = "archie-evedon-2026"  # Example: "ben-stevo-test-rides"
EVENT_SCHEMA_PATH = ANALYSIS_DIR / "event schema" / "event_schema.yaml"

# Include Study Set groupings as grouped scope entities. Sessions that are not
# in a grouping remain available as individual session entities.
INCLUDE_STUDY_SET_GROUPINGS = True

print(f"Analysis package root: {ANALYSIS_DIR}")
print(f"Libraries root: {LIBRARIES_ROOT}")


Analysis package root: C:\Users\benco\dev\BODAQS\analysis
Libraries root: C:\Users\benco\OneDrive\BODAQS-data


## 2. Load Study Set Scope

If `TARGET_STUDY_SET_ID` is blank, this cell lists available Study Sets for the configured library and stops. Set the ID, then rerun from this cell.


In [2]:
from bodaqs_analysis.library_api import InvalidStudySetError, LibraryAdapter
from bodaqs_analysis.widgets.event_schema_resolution import resolve_event_schema_for_selection
from bodaqs_analysis.widgets.loaders import make_session_loader

adapter = LibraryAdapter(LIBRARIES_ROOT)
libraries = {item["library_id"]: item for item in adapter.list_libraries()}
if LIBRARY_ID not in libraries:
    available = ", ".join(sorted(libraries)) or "none found"
    raise ValueError(f"Library {LIBRARY_ID!r} was not found. Available libraries: {available}")

available_study_sets = adapter.list_study_sets(library_id=LIBRARY_ID)
if not TARGET_STUDY_SET_ID.strip():
    if available_study_sets:
        display(pd.DataFrame(available_study_sets))
    else:
        print("No Study Sets found for this library.")
    raise ValueError("Set TARGET_STUDY_SET_ID to a saved Study Set ID, then rerun this cell.")

try:
    study_set_bridge = adapter.study_set_to_selection_snapshot(
        LIBRARY_ID,
        TARGET_STUDY_SET_ID.strip(),
        include_groupings=INCLUDE_STUDY_SET_GROUPINGS,
    )
except InvalidStudySetError as exc:
    message = str(exc)
    if "one-library Study Sets" in message:
        raise RuntimeError(
            "This pilot notebook currently supports one-library Study Sets only. "
            "Open a Study Set whose sessions all belong to LIBRARY_ID, or wait for the multi-library notebook bridge."
        ) from exc
    raise

sel = study_set_bridge["selector_handle"]
store = study_set_bridge["store"]
key_to_ref = sel["get_key_to_ref"]()
events_index_df = sel["get_events_index_df"]()
session_loader = make_session_loader(store=store, key_to_ref=key_to_ref)

schema_resolution = resolve_event_schema_for_selection(
    sel,
    fallback_schema_path=EVENT_SCHEMA_PATH,
)
schema = schema_resolution.schema

print(f"Loaded Study Set: {study_set_bridge['display_name']} ({study_set_bridge['study_set_id']})")
print(f"Sessions in scope: {len(key_to_ref)}")
print(f"Event schema source: {schema_resolution.source}")
if schema_resolution.sha256:
    print(f"Event schema sha256: {schema_resolution.sha256}")
for warning in schema_resolution.warnings:
    print(f"WARNING: {warning}")
display(events_index_df)


Loaded Study Set: Archie-Evedon-2026 (archie-evedon-2026)
Sessions in scope: 3
Event schema source: frozen_artifacts
Event schema sha256: be7931dff3eddce88fb4e5aefe7ff3ba613789d37408a2ce58ca754477ada3fd


,session_key,run_id,session_id
0,archie-mega-local_260614_172144::260613_111401,archie-mega-local_260614_172144,260613_111401
1,archie-mega-local_260614_172144::260613_103335,archie-mega-local_260614_172144,260613_103335
2,archie-mega-local_260614_172144::260613_100618,archie-mega-local_260614_172144,260613_100618


## 3. Signal Histogram

Explore frequency distributions for session signal series in the Study Set scope.


In [3]:
from bodaqs_analysis.widgets.signal_histogram_widget import make_signal_histogram_rebuilder

hist = make_signal_histogram_rebuilder(sel=sel)
display(hist["out"])


Output()

## 4. Event Browser

Inspect signal data around detected events.


In [4]:
from bodaqs_analysis.widgets.event_browser import make_event_browser_rebuilder

if not key_to_ref:
    raise ValueError("The loaded Study Set has no sessions.")

browser = make_event_browser_rebuilder(sel=sel, schema=schema)
display(browser["out"])


Output()

## 5. Metric Scatter Plot

Compare two metrics for a selected event type.


In [5]:
from bodaqs_analysis.widgets.metric_scatter_widget import make_metric_scatter_rebuilder

scatter = make_metric_scatter_rebuilder(sel=sel, schema=schema)
display(scatter["out"])


Output()

## 6. Metric Histogram

Explore distributions of event metrics across the Study Set scope.


In [6]:
from bodaqs_analysis.widgets.metric_histogram_widget import make_metric_histogram_rebuilder

mhist = make_metric_histogram_rebuilder(sel=sel, schema=schema)
display(mhist["out"])


Output()

## Notes

Because this notebook has a fixed persisted scope rather than a live selector, there are no selector refresh hooks. If you change `TARGET_STUDY_SET_ID`, rerun the load cell and widget cells.
